# 01 — Explore and select CB2 Ki measurements

This notebook rebuilds **the first three stages** of the rough-draft `01_data_exploration` notebook:

1. Load and inspect the raw ChEMBL download.
2. Select Ki measurements from direct human CB2 binding assays.
3. Check values and record quality, keeping review groups visible.

Run the cells from top to bottom. They read the latest **complete** acquisition in this repository's `data/raw` folder. No network call or extra Python package is needed. This notebook keeps results in memory; it does not change the raw files or export a curated dataset yet.

The rough draft uses pandas tables. This version uses lists of records (Python dictionaries) to make the filters and counts visible. It preserves the draft's scientific selection rules and stops **before** RDKit structure checks, pKi conversion, and aggregation. A record reaching the end is a candidate for those later steps, not a finalized model row.

## Stage 1 — Load the raw download

`00_data_acquisition` creates a dated directory and writes `acquisition_manifest.json` only after its checks pass. Choose the newest directory with that file. This prevents accidentally reading a partial download. The same three JSON tables as in the rough draft are loaded: activities, assays, and documents.

In [1]:
from pathlib import Path
from collections import Counter
import json
import math

# Accept either the repository root or its notebooks folder as the working directory.
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
assert (project_root / "notebooks").is_dir(), "Run this notebook from the repository or notebooks folder."

# A manifest is the completion marker written by notebook 00.
raw_root = project_root / "data" / "raw"
complete_downloads = sorted(raw_root.glob("chembl_cb2_*/acquisition_manifest.json"))
if not complete_downloads:
    raise FileNotFoundError("No complete download found. Run 00_data_acquisition first.")

manifest_path = complete_downloads[-1]
data_folder = manifest_path.parent
with manifest_path.open(encoding="utf-8") as file:
    manifest = json.load(file)

# Each JSON table is a list of dictionaries: one dictionary per ChEMBL record.
with (data_folder / "chembl_cb2_activity.json").open(encoding="utf-8") as file:
    activities = json.load(file)
with (data_folder / "chembl_cb2_assay.json").open(encoding="utf-8") as file:
    assays = json.load(file)
with (data_folder / "chembl_cb2_documents.json").open(encoding="utf-8") as file:
    documents = json.load(file)

# Verify that the files match the acquisition's recorded counts.
assert len(activities) == manifest["activity_records"]
assert len(assays) == manifest["assay_records"]
assert len(documents) == manifest["document_records"]
assert manifest["target_chembl_id"] == "CHEMBL253"

print("Source:", data_folder)
print("ChEMBL release:", manifest["chembl_version"])
print(f"Activities: {len(activities):,}")
print(f"Assays: {len(assays):,}")
print(f"Documents: {len(documents):,}")

Source: /home/justin/code/cb2-affinity/data/raw/chembl_cb2_20260922T165756862384Z
ChEMBL release: ChEMBL_37
Activities: 22,523
Assays: 1,551
Documents: 653


### See what was downloaded

`standard_type` names the measured endpoint, such as Ki or EC50. `standard_relation` tells us whether a number is exact (`=`) or a limit (`>` or `<`). At this point we only count them; we have not excluded anything.

In [2]:
# Show a few fields from the first activity, without printing its entire nested record.
example_fields = [
    "activity_id", "molecule_chembl_id", "canonical_smiles",
    "standard_type", "standard_relation", "standard_value",
    "standard_units", "assay_chembl_id", "document_chembl_id",
]
print("Example activity:")
for field in example_fields:
    print(f"  {field}: {activities[0].get(field)}")

# Count unique nonempty molecule IDs and the different endpoint types.
molecule_ids = {row.get("molecule_chembl_id") for row in activities if row.get("molecule_chembl_id")}
endpoint_counts = Counter(row.get("standard_type") for row in activities)
relation_counts = Counter(row.get("standard_relation") for row in activities)

print(f"\nDistinct molecule IDs: {len(molecule_ids):,}")
print("\nMeasurement types:")
for endpoint, count in endpoint_counts.most_common():
    print(f"  {str(endpoint):16} {count:>6,}")
print("\nMeasurement relations:")
for relation, count in relation_counts.most_common():
    print(f"  {str(relation):16} {count:>6,}")

Example activity:
  activity_id: 35234
  molecule_chembl_id: CHEMBL158502
  canonical_smiles: N=C(NS(=O)(=O)c1ccc(F)cc1)N1CC(c2ccccc2)C(c2ccc(Cl)cc2)=N1
  standard_type: Ki
  standard_relation: >
  standard_value: 1000.0
  standard_units: nM
  assay_chembl_id: CHEMBL659744
  document_chembl_id: CHEMBL1148395

Distinct molecule IDs: 11,399

Measurement types:
  Ki                6,571
  EC50              5,503
  IC50              2,093
  Activity          1,845
  Emax              1,570
  kon               1,549
  k_off             1,549
  Inhibition        1,091
  Efficacy            356
  Imax                 77
  pEC50                55
  Intrinsic activity     35
  Displacement         30
  Kieq                 25
  % Ctrl               25
  Log Ki               20
  logEC50              19
  Kb                   14
  Kd                   14
  Ratio                13
  Bmax                 12
  pKi                   8
  % Inhibition of Control Specific Binding (Mean n=2)      8
  de

## Stage 2 — Choose direct human binding Ki records

First select `standard_type == "Ki"`. Then connect each activity to its assay by `assay_chembl_id`. An assay should have one metadata record, but it may support many activities. We check that no Ki activity loses its assay metadata.

The draft's three sequential assay rules are retained: binding assay type `B`, ChEMBL confidence score `9` for direct target assignment, and assay organism `Homo sapiens`. Requiring an explicitly human **assay** entry is stricter than relying only on the human target record. Counts show the effect of each rule.

In [3]:
# Ki is a binding-affinity endpoint. Keep all other raw records untouched.
ki = [row for row in activities if row.get("standard_type") == "Ki"]
print(f"Ki activities: {len(ki):,}")
print("Distinct Ki molecule IDs:", len({row.get("molecule_chembl_id") for row in ki if row.get("molecule_chembl_id")}))

# Build a lookup by assay ID. The uniqueness check protects the many-to-one join.
assays_by_id = {}
for assay in assays:
    assay_id = assay["assay_chembl_id"]
    assert assay_id not in assays_by_id, "Repeated assay ID: review the raw download."
    assays_by_id[assay_id] = assay

# Copy each activity, then add the assay fields needed for selection and review.
ki_with_assays = []
for activity in ki:
    assay_id = activity.get("assay_chembl_id")
    assert assay_id in assays_by_id, f"Missing assay metadata for {assay_id}."
    assay = assays_by_id[assay_id]
    row = activity.copy()
    row["assay_organism"] = assay.get("assay_organism")
    row["confidence_score"] = assay.get("confidence_score")
    row["confidence_description"] = assay.get("confidence_description")
    row["relationship_type"] = assay.get("relationship_type")
    row["assay_description"] = assay.get("description")
    ki_with_assays.append(row)

assert len(ki_with_assays) == len(ki), "The assay join changed the activity count."
print(f"Ki activities matched to assays: {len(ki_with_assays):,}")

Ki activities: 6,571
Distinct Ki molecule IDs: 5350
Ki activities matched to assays: 6,571


### Inspect the assay mix

The counts below show where the Ki records occur. A distinct assay is an experiment identifier; it is different from the number of activity measurements. The example list highlights assays excluded by at least one rule, including their descriptions, before filtering them out.

In [4]:
# Count measurements by assay type, organism, and confidence score.
assay_groups = Counter(
    (row.get("assay_type"), row.get("assay_organism"), row.get("confidence_score"))
    for row in ki_with_assays
)
print("Assay type | organism | confidence | measurements")
for group, count in assay_groups.most_common():
    print(f"{str(group[0]):>10} | {str(group[1]):>16} | {str(group[2]):>10} | {count:>6,}")

# Show each excluded assay at most once so descriptions are not repeated.
excluded_assays = {}
for row in ki_with_assays:
    meets_rules = (
        row.get("assay_type") == "B"
        and row.get("confidence_score") == 9
        and row.get("assay_organism") == "Homo sapiens"
    )
    if not meets_rules:
        excluded_assays[row["assay_chembl_id"]] = row

print(f"\nDistinct assays needing review: {len(excluded_assays):,}")
for row in list(excluded_assays.values())[:8]:
    print(row["assay_chembl_id"], "|", row.get("assay_type"), "|",
          row.get("confidence_score"), "|", row.get("assay_organism"), "|",
          str(row.get("assay_description"))[:120])

Assay type | organism | confidence | measurements
         B |     Homo sapiens |          9 |  5,825
         B |             None |          8 |    486
         B |     Homo sapiens |          8 |    186
         A |     Homo sapiens |          9 |     44
         F |     Homo sapiens |          9 |     24
         B |  Cavia porcellus |          8 |      5
         F |             None |          8 |      1

Distinct assays needing review: 58
CHEMBL659744 | B | 8 | None | Affinity to displace CP-55940 binding from Cannabinoid receptor 2 of human expressed in CHO cells
CHEMBL659745 | B | 8 | None | Affinity to displace CP-55940 binding from Cannabinoid receptor 2 of human expressed in CHO cells; Not determined
CHEMBL658954 | B | 8 | None | Binding affinity against G protein coupled human Cannabinoid receptor 2 using [3H]CP-55940 in HEK293 EBNA transfected ce
CHEMBL658953 | B | 8 | None | Binding affinity of compound was determined against to human cannabinoid receptor 2 in chinese ha

In [5]:
# Apply one criterion at a time to make the loss at each step visible.
ki_binding = [row for row in ki_with_assays if row.get("assay_type") == "B"]
ki_direct = [row for row in ki_binding if row.get("confidence_score") == 9]
ki_human = [row for row in ki_direct if row.get("assay_organism") == "Homo sapiens"]

assay_stages = [
    ("All Ki activities", ki_with_assays),
    ("Binding assays (B)", ki_binding),
    ("Direct target assignment (9)", ki_direct),
    ("Human assay organism", ki_human),
]
previous_count = None
print("Stage | measurements | molecule IDs | assays | removed at step")
for name, rows in assay_stages:
    molecule_count = len({row.get("molecule_chembl_id") for row in rows if row.get("molecule_chembl_id")})
    assay_count = len({row["assay_chembl_id"] for row in rows})
    removed = 0 if previous_count is None else previous_count - len(rows)
    print(f"{name}: {len(rows):,} | {molecule_count:,} | {assay_count:,} | {removed:,}")
    previous_count = len(rows)

assert len(ki_human) <= len(ki_direct) <= len(ki_binding) <= len(ki_with_assays)

Stage | measurements | molecule IDs | assays | removed at step
All Ki activities: 6,571 | 5,350 | 432 | 0
Binding assays (B): 6,502 | 5,295 | 422 | 69
Direct target assignment (9): 5,825 | 4,813 | 374 | 677
Human assay organism: 5,825 | 4,813 | 374 | 0


## Stage 3 — Check measurement values and record quality

The assay rules identify relevant experiments, but some rows may still lack essential fields or a usable numeric Ki. The following checks match the draft: an activity ID, molecule ID, SMILES, and assay ID must be present; Ki must be numeric, finite, positive, and reported in `nM`.

Keep the original `standard_value`. Add a working `ki_nm` number only after checking the unit. Empty text is treated as missing. This stage does **not** validate SMILES chemically; that comes later.

In [6]:
text_fields = [
    "molecule_chembl_id", "canonical_smiles", "assay_chembl_id",
    "document_chembl_id", "standard_relation", "standard_units",
    "data_validity_comment",
]

ki_work = []
for original in ki_human:
    row = original.copy()
    for field in text_fields:
        value = row.get(field)
        if isinstance(value, str):
            value = value.strip()
            row[field] = value if value else None

    # A bad or missing number becomes None; it will be set aside below.
    try:
        row["ki_nm"] = float(row["standard_value"])
    except (TypeError, ValueError):
        row["ki_nm"] = None
    ki_work.append(row)

important_fields = [
    "activity_id", "molecule_chembl_id", "canonical_smiles",
    "assay_chembl_id", "document_chembl_id", "standard_relation",
    "standard_value", "standard_units", "data_validity_comment",
    "potential_duplicate",
]
print("Missing values after trimming blank text:")
for field in important_fields:
    missing = sum(row.get(field) is None for row in ki_work)
    print(f"  {field:24} {missing:>6,} ({missing / len(ki_work):.1%})")

for field in ["standard_relation", "standard_units", "data_validity_comment", "potential_duplicate"]:
    print(f"\n{field}:")
    for value, count in Counter(row.get(field) for row in ki_work).most_common():
        print(f"  {str(value):24} {count:>6,}")

Missing values after trimming blank text:
  activity_id                   0 (0.0%)
  molecule_chembl_id            0 (0.0%)
  canonical_smiles              0 (0.0%)
  assay_chembl_id               0 (0.0%)
  document_chembl_id            0 (0.0%)
  standard_relation           228 (3.9%)
  standard_value              228 (3.9%)
  standard_units              226 (3.9%)
  data_validity_comment     5,820 (99.9%)
  potential_duplicate           0 (0.0%)

standard_relation:
  =                         4,576
  >                           990
  None                        228
  <                            16
  >=                           12
  ~                             2
  <=                            1

standard_units:
  nM                        5,599
  None                        226

data_validity_comment:
  None                      5,820
  Potential transcription error      4
  Outside typical range         1

potential_duplicate:
  0                         5,315
  1              

In [7]:
required_fields = ["activity_id", "molecule_chembl_id", "canonical_smiles", "assay_chembl_id"]
failed_checks = Counter()
ki_usable = []
basic_review = []

for row in ki_work:
    # Each named check answers one question about this measurement.
    essential_present = all(row.get(field) is not None for field in required_fields)
    numeric_and_finite = row["ki_nm"] is not None and math.isfinite(row["ki_nm"])
    positive_ki = numeric_and_finite and row["ki_nm"] > 0
    units_are_nm = row.get("standard_units") == "nM"

    checks = {
        "essential fields present": essential_present,
        "numeric and finite": numeric_and_finite,
        "positive Ki": positive_ki,
        "units are nM": units_are_nm,
    }
    for name, passed in checks.items():
        if not passed:
            failed_checks[name] += 1

    # Rows can fail more than one check, so failure counts do not sum to exclusions.
    if all(checks.values()):
        ki_usable.append(row)
    else:
        basic_review.append(row)

print("Rows failing each check:")
for name in ["essential fields present", "numeric and finite", "positive Ki", "units are nM"]:
    print(f"  {name:25} {failed_checks[name]:>6,}")
print(f"Retained: {len(ki_usable):,}; set aside: {len(basic_review):,}")
assert len(ki_usable) + len(basic_review) == len(ki_work)

Rows failing each check:
  essential fields present       0
  numeric and finite           228
  positive Ki                  228
  units are nM                 226
Retained: 5,597; set aside: 228


### Keep exact Ki measurements

A relation of `=` is an exact value. Relations such as `>` and `<` are **limits**: treating a bound as an exact training target would misstate the measurement. Keep limits and unusual or missing relations in separate review lists. No conversion or substitution is attempted here.

In [8]:
ki_exact = []
ki_limits = []
relation_review = []
for row in ki_usable:
    relation = row.get("standard_relation")
    if relation == "=":
        ki_exact.append(row)
    elif relation in {">", ">=", "<", "<="}:
        ki_limits.append(row)
    else:
        relation_review.append(row)

print(f"Exact Ki: {len(ki_exact):,}")
print(f"Limit values: {len(ki_limits):,}")
print(f"Other or missing relations: {len(relation_review):,}")
assert len(ki_exact) + len(ki_limits) + len(relation_review) == len(ki_usable)

Exact Ki: 4,576
Limit values: 1,019
Other or missing relations: 2


### Apply ChEMBL's quality flags

The draft accepts a blank validity comment or `Manually validated`. Other comments are set aside for review. It also requires `potential_duplicate` to be explicitly `0`; an unknown or flagged value does not pass. This is a conservative dataset choice, not proof that other measurements are wrong.

In [9]:
ki_quality = []
validity_review = []
for row in ki_exact:
    if row.get("data_validity_comment") in {None, "Manually validated"}:
        ki_quality.append(row)
    else:
        validity_review.append(row)

ki_unflagged = []
duplicate_flag_review = []
for row in ki_quality:
    # ChEMBL's JSON value may be integer 0 or text "0".
    if str(row.get("potential_duplicate")) == "0":
        ki_unflagged.append(row)
    else:
        duplicate_flag_review.append(row)

print(f"Set aside for validity comments: {len(validity_review):,}")
print(f"Set aside for duplicate or unknown flag: {len(duplicate_flag_review):,}")
print(f"Remaining: {len(ki_unflagged):,}")
assert len(ki_quality) + len(validity_review) == len(ki_exact)
assert len(ki_unflagged) + len(duplicate_flag_review) == len(ki_quality)

Set aside for validity comments: 5
Set aside for duplicate or unknown flag: 490
Remaining: 4,081


### Remove identical records and check activity IDs

Two identical copies of one record should count once. JSON serialization gives us a stable comparison even when a field contains a nested list or dictionary. After removing identical copies, a repeated `activity_id` with different content would be a conflict and stop the notebook for investigation.

In [10]:
seen_records = set()
ki_records = []
duplicate_records = []
for row in ki_unflagged:
    # Sort dictionary keys so key order does not affect the comparison.
    signature = json.dumps(row, sort_keys=True, ensure_ascii=False)
    if signature in seen_records:
        duplicate_records.append(row)
    else:
        seen_records.add(signature)
        ki_records.append(row)

activity_id_counts = Counter(row["activity_id"] for row in ki_records)
activity_id_conflicts = [row for row in ki_records if activity_id_counts[row["activity_id"]] > 1]
print(f"Identical copies removed: {len(duplicate_records):,}")
print(f"Activity IDs with conflicting records: {len(set(row['activity_id'] for row in activity_id_conflicts)):,}")
assert not activity_id_conflicts, "One activity ID describes different records; investigate before continuing."
print(f"Unique activity records retained: {len(ki_records):,}")

Identical copies removed: 0
Activity IDs with conflicting records: 0
Unique activity records retained: 4,081


### Review the full selection audit

Each line is cumulative. The retained `ki_records` still represent individual **measurements**, so a molecule or structure can occur more than once. The next stage will parse molecular structures, calculate pKi, and investigate repeated measurements before choosing any one-row-per-structure modeling target.

In [11]:
cleaning_stages = [
    ("Human binding Ki before cleaning", ki_work),
    ("Essential fields and valid nM Ki", ki_usable),
    ("Exact Ki measurements", ki_exact),
    ("Acceptable validity status", ki_quality),
    ("No potential-duplicate flag", ki_unflagged),
    ("Unique activity records", ki_records),
]
previous_count = None
print("Stage | measurements | removed at step")
for name, rows in cleaning_stages:
    removed = 0 if previous_count is None else previous_count - len(rows)
    print(f"{name}: {len(rows):,} | {removed:,}")
    previous_count = len(rows)

# These groups account for every measurement entering the cleaning stage.
assert (
    len(ki_records) + len(duplicate_records) + len(duplicate_flag_review)
    + len(validity_review) + len(ki_limits) + len(relation_review)
    + len(basic_review) == len(ki_work)
)
print("Every selected measurement is accounted for.")

Stage | measurements | removed at step
Human binding Ki before cleaning: 5,825 | 0
Essential fields and valid nM Ki: 5,597 | 228
Exact Ki measurements: 4,576 | 1,021
Acceptable validity status: 4,571 | 5
No potential-duplicate flag: 4,081 | 490
Unique activity records: 4,081 | 0
Every selected measurement is accounted for.


## Stop here for review

The scientific choices to review before continuing are the strict assay filters, exact values in nM only, accepted validity comments, and the explicit zero duplicate flag. The raw data remains available for revisiting any of these choices. No structure validation, pKi conversion, or curated export has happened yet.